# Construir el dataset combinado del equipo

Antes de correr:
1. **Add Input** → agrega las fuentes: `moltean/fruits` (Fruits-360), `swoyam2609/fresh-and-stale-classification`, `sergeynesteruk/packed-fruits-and-vegetables-recognition-benchmark`, `misrakahmed/vegetable-image-dataset` y los datasets privados del equipo (vegetales de Bangladesh y fotos del equipo). Si falta alguna, se omite con un aviso.
2. **Settings** → Internet: **On** (también sirve para clonar el Grocery Store Dataset de GitHub). No hace falta GPU.

Al terminar, el dataset queda en `/kaggle/working/dataset` con `train/`, `test/`, `manifest.csv`, `conteos.csv` y `reporte.json`.

In [ ]:
import os, subprocess

REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    r = subprocess.run(["git", "clone", "https://github.com/11909/Clasificador-fruta-verdura.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo. Activa Settings → Internet en el panel derecho "
                           "(Kaggle pide verificar tu teléfono) y vuelve a correr esta celda.\n" + r.stderr)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"])  # trae la última versión de main
%cd {REPO}
!pip install -q -r requirements.txt

## 0. Fuentes que no están en Kaggle

El Grocery Store Dataset (fotos en supermercados, licencia MIT) se clona de GitHub.

In [ ]:
import os
GROCERY = "/kaggle/working/fuentes/GroceryStoreDataset"
if not os.path.isdir(GROCERY):
    !git clone -q --depth 1 https://github.com/marcusklasson/GroceryStoreDataset.git {GROCERY}

## 1. Revisar la estructura de cada fuente

Con esto se ve en qué carpetas están los splits y en qué nivel está la clase. Si no coincide con `configs/dataset.yaml`, corrige `splits` y `nivel_clase` en la celda siguiente.

In [ ]:
import glob
from src.build_dataset import explorar

for raiz in sorted(glob.glob("/kaggle/input/*")):
    explorar(raiz, profundidad=4)
    print()

## 2. Ajustar la config (si hace falta)

In [ ]:
import yaml

cfg = yaml.safe_load(open("configs/dataset.yaml"))
# Ejemplos de ajuste:
# cfg["fuentes"][1]["splits"]["train"] = "/kaggle/input/fresh-and-stale-classification/otra/ruta/Train"
# cfg["fuentes"][2]["nivel_clase"] = 1
# cfg["fuentes"] = [f for f in cfg["fuentes"] if f["nombre"] != "packeat"]   # quitar una fuente
for f in cfg["fuentes"]:
    print(f["nombre"], f["splits"], "nivel_clase =", f.get("nivel_clase", 0))

## 3. Construir

In [ ]:
import json
from src.build_dataset import construir

reporte = construir(cfg)
print(json.dumps(reporte, indent=2, ensure_ascii=False))

## Desbalance de clases

Imágenes por etiqueta y las clases de cada etiqueta, de más a menos.

In [ ]:
import pandas as pd

conteos = pd.read_csv(f"{cfg['salida']}/conteos.csv")
conteos["total"] = conteos["train"] + conteos["test"]
display(conteos.groupby("etiqueta")[["train", "test", "total"]].sum())
por_clase = conteos.pivot_table(index=["etiqueta", "clase"], columns="fuente", values="total", aggfunc="sum", fill_value=0)
por_clase["total"] = por_clase.sum(axis=1)
display(por_clase.sort_values(["etiqueta", "total"], ascending=[True, False]))

## 4. Publicarlo como dataset del equipo

1. **Save Version → Save & Run All** para guardar la salida.
2. En la versión guardada, pestaña **Output** → **New Dataset**. Déjalo **privado** y compártelo con el equipo (**Settings → Sharing**).
3. Cada vez que cambien fuentes o el mapeo, corran de nuevo y usen **New Version** sobre el mismo dataset.

El notebook de entrenamiento detecta solo el dataset del equipo (por su `manifest.csv`) cuando está adjunto.